# Fine-tuning Qwen3-1.7B for Bash Reconstruction

Train Qwen3-1.7B (text-only) to reconstruct shell commands from voice dictation.

- **Model:** Qwen3-1.7B (text-only, thinking mode toggleable)
- **Method:** LoRA via Unsloth on free Colab T4
- **Data:** 5,044 train / 630 val / 630 test from [arach/training-lab](https://huggingface.co/arach/training-lab)
- **Task:** Dictated text → intended syntax

Part of the [Teaching a Tiny Model to Hear Bash](https://usetalkie.com) series.

## 1. Install Dependencies

In [ ]:
try:
    import unsloth
    print("✓ Already installed, skipping")
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "unsloth", "unsloth_zoo", "huggingface_hub", "datasets"], capture_output=True)
    print("✓ Installed")

## 2. Load Model

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-1.7B",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,
)

## 3. Configure LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    max_seq_length=max_seq_length,
)

model.print_trainable_parameters()

## 4. Load Training Data

Data lives on HuggingFace at `arach/training-lab`.
- 5,044 training examples
- Chat format: system → user (dictated) → assistant (syntax)

In [ ]:
from huggingface_hub import hf_hub_download
from datasets import load_dataset

for split in ["train", "valid", "test"]:
    hf_hub_download(
        repo_id="arach/training-lab",
        filename=f"training/data/bash-v2/minimal/{split}.jsonl",
        local_dir="./data",
        repo_type="model",
    )

train_dataset = load_dataset("json", data_files="./data/training/data/bash-v2/minimal/train.jsonl", split="train")
val_dataset = load_dataset("json", data_files="./data/training/data/bash-v2/minimal/valid.jsonl", split="train")

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")
print(f"Example: {train_dataset[0]}")

## 5. Format for Chat Template

Qwen3 is text-only — `apply_chat_template` works directly.
We disable thinking with `enable_thinking=False`.

In [ ]:
def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )
    return {"text": text}

train_dataset = train_dataset.map(format_chat)
val_dataset = val_dataset.map(format_chat)

print(train_dataset[0]["text"])

## 6. Train

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    formatting_func=lambda example: [example["text"]],
    args=SFTConfig(
        max_seq_length=max_seq_length,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=20,
        num_train_epochs=3,
        logging_steps=25,
        eval_strategy="steps",
        eval_steps=200,
        save_steps=200,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        learning_rate=1e-4,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="outputs_qwen3_1.7b",
    ),
)

trainer_stats = trainer.train()
print(f"\nTraining time: {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"Peak memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

# Auto-save adapter immediately after training
model.save_pretrained("qwen3-1.7b-bash-v1")
tokenizer.save_pretrained("qwen3-1.7b-bash-v1")
print("Adapter saved to qwen3-1.7b-bash-v1/")

## 7. Quick Inference Test

In [ ]:
FastLanguageModel.for_inference(model)

test_cases = [
    "git space push space dash u space origin space main",
    "find dot dash name star dot txt",
    "docker space run space dash dash rm space dash p space eight zero eight zero colon eight zero space nginx",
    "chmod space zero seven five five space slash usr slash local slash bin slash deploy dot sh",
    "ssh space dash i space tilde slash dot ssh slash id underscore rsa space root at one nine two dot one six eight dot one dot one",
]

for dictated in test_cases:
    messages = [
        {"role": "system", "content": "Reconstruct the intended syntax from the dictated text. Output only the result."},
        {"role": "user", "content": dictated},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", enable_thinking=False,
    ).to(model.device)

    outputs = model.generate(
        input_ids=inputs, max_new_tokens=64, temperature=0.0,
        do_sample=False, use_cache=True,
    )
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    print(f"IN:  {dictated}")
    print(f"OUT: {response}")
    print()

## 8. Full Evaluation

In [ ]:
from difflib import SequenceMatcher
import time

test_dataset = load_dataset("json", data_files="./data/training/data/bash-v2/minimal/test.jsonl", split="train")

results = {"exact": 0, "near": 0, "partial": 0, "wrong": 0}
total_time = 0
errors = []

for i, example in enumerate(test_dataset):
    msgs = example["messages"]
    dictated = msgs[1]["content"]
    expected = msgs[2]["content"]

    messages = [
        {"role": "system", "content": msgs[0]["content"]},
        {"role": "user", "content": dictated},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", enable_thinking=False,
    ).to(model.device)

    t0 = time.time()
    outputs = model.generate(
        input_ids=inputs, max_new_tokens=64, temperature=0.0,
        do_sample=False, use_cache=True,
    )
    t1 = time.time()
    total_time += (t1 - t0)

    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True).strip()
    similarity = SequenceMatcher(None, response, expected).ratio()

    if response == expected:
        results["exact"] += 1
    elif similarity >= 0.9:
        results["near"] += 1
    elif similarity >= 0.7:
        results["partial"] += 1
    else:
        results["wrong"] += 1
        errors.append((dictated[:60], expected[:60], response[:60]))

    if (i + 1) % 10 == 0:
        elapsed = total_time / (i + 1)
        print(f"  {i+1}/{len(test_dataset)} | avg {elapsed:.1f}s/ex | exact so far: {results['exact']}/{i+1}")

n = len(test_dataset)
print(f"\n{'='*50}")
print(f"QWEN3-1.7B EVALUATION ({n} cases)")
print(f"{'='*50}")
print(f"Exact match:    {results['exact']:4d} / {n} ({results['exact']/n*100:.1f}%)")
print(f"Near (>90%):    {results['near']:4d} / {n} ({results['near']/n*100:.1f}%)")
print(f"Partial (70-90): {results['partial']:3d} / {n} ({results['partial']/n*100:.1f}%)")
print(f"Wrong (<70%):    {results['wrong']:3d} / {n} ({results['wrong']/n*100:.1f}%)")
print(f"\nEffective accuracy: {(results['exact']+results['near'])/n*100:.1f}%")
print(f"Avg inference time: {total_time/n:.2f}s")
print(f"Peak memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

if errors:
    print(f"\nSample errors ({len(errors)} total):")
    for d, e, r in errors[:5]:
        print(f"  IN:  {d}")
        print(f"  EXP: {e}")
        print(f"  GOT: {r}")
        print()

## 9. Save Adapter

In [ ]:
model.save_pretrained("qwen3-1.7b-bash-v1")
tokenizer.save_pretrained("qwen3-1.7b-bash-v1")
print("Adapter saved to qwen3-1.7b-bash-v1/")

## 10. Push Adapter to HuggingFace

In [ ]:
# Uncomment and set your token to push
# from huggingface_hub import login
# login(token="hf_...")
#
# model.push_to_hub("arach/qwen3-1.7b-bash-v1")
# tokenizer.push_to_hub("arach/qwen3-1.7b-bash-v1")
# print("Pushed to HuggingFace!")